# Water Quality Prediction: Benchmark Notebook 

## Challenge Overview

<p align="justify">
Welcome to the EY AI & Data Challenge 2026!  
The objective of this challenge is to build a robust <strong> machine learning model </strong>capable of predicting water quality across various river locations in South Africa. In addition to accurate predictions, the model should also identify and emphasize the key factors that significantly influence water quality.
</p>

<p align="justify">
Participants will be provided with a dataset containing three water quality parameters <strong>Total Alkalinity</strong>, <strong>Electrical Conductance</strong>, and <strong>Dissolved Reactive Phosphorus</strong> collected between 2011 and 2015 from approximately 200 river locations across South Africa. Each data point includes the geographic coordinates (latitude and longitude) of the sampling site, the date of collection, and the corresponding water quality measurements.
</p>

<p align="justify">
Using this dataset, participants are expected to build a machine learning model to predict water quality parameters for a separate validation dataset, which includes locations from different regions not present in the training data. The challenge also encourages participants to explore feature importance and provide insights into the factors most strongly associated with variations in water quality.
</p>

<p align="justify">
This challenge is designed for participants with varying levels of experience in data science, remote sensing, and environmental analytics. It offers a valuable opportunity to apply machine learning techniques to real-world environmental data and contribute to advancing water quality monitoring using artificial intelligence.
</p>


<b>About the Notebook: </b><p align="justify"> <p>

<p align="justify"> In this notebook, we demonstrate a basic workflow that serves as a foundation for the challenge. The model has been developed to predict <b>water quality parameters</b> using features derived from the <b>Landsat</b> and <b>TerraClimate</b> datasets. Specifically, four spectral bands—<b>SWIR22</b> (Shortwave Infrared 2), <b>NIR</b> (Near Infrared), <b>Green</b>, and <b>SWIR16</b> (Shortwave Infrared 1)—were utilized from Landsat, along with derived spectral indices such as <b>NDMI</b> (Normalized Difference Moisture Index) and <b>MNDWI</b> (Modified Normalized Difference Water Index). In addition, the <b>PET</b> (Potential Evapotranspiration) variable was incorporated from the <b>TerraClimate</b> dataset to account for climatic influences on water quality. </p> 

<p align="justify"> The dataset spans a five-year period from <b>2011 to 2015</b>. Using <b>API-based data extraction</b> methods, both Landsat and TerraClimate features were retrieved directly from the <a href="https://planetarycomputer.microsoft.com/">Microsoft Planetary Computer</a> portal. These combined spectral, index-based, and climatic features were used as predictors in a regression model to estimate three key water quality parameters: <b>Total Alkalinity (TA)</b>, <b>Electrical Conductance (EC)</b>, and <b>Dissolved Reactive Phosphorus (DRP)</b>. 

</p> <p align="justify"> Please note that this notebook serves only as a starting point. Several assumptions were made during the data extraction and model development process, which you may find opportunities to improve upon. Participants are encouraged to explore additional features, enhance preprocessing techniques, or experiment with different regression algorithms to optimize predictive performance. </p>

## Load In Dependencies

To run this demonstration notebook, you will need to have the following packages imported below installed. This may take some time.  

In [16]:
# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Feature preprocessing and data splitting
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.spatial import cKDTree

# Machine Learning
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

# Planetary Computer tools for STAC API access and authentication
import pystac_client
import planetary_computer as pc
from odc.stac import stac_load
from pystac.extensions.eo import EOExtension as eo

from datetime import date
from tqdm import tqdm
import os 

## Response Variable

<p align="justify">
Before building the model, we first load the <b>water quality training dataset</b>. The curated dataset contains samples collected from various monitoring stations across the study region. Each record includes the geographical coordinates (Latitude and Longitude), the sample collection date, and the corresponding <b>measured values</b> for the three key water quality parameters — <b>Total Alkalinity (TA)</b>, <b>Electrical Conductance (EC)</b>, and <b>Dissolved Reactive Phosphorus (DRP)</b>.
</p>


In [17]:
Water_Quality_df=pd.read_csv('../Data/water_quality_training_dataset.csv')
Water_Quality_df.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0


## Predictor Variables

<p align="justify">
Now that we have our water quality dataset, the next step is to gather the predictor variables from the <b>Landsat</b> and <b>TerraClimate</b> datasets. In this notebook, we demonstrate how to <b>load previously extracted satellite and climate data</b> from separate files, rather than performing the extraction directly, which allows for a smoother and faster experience. Participants can refer to the dedicated extraction notebooks—one for Landsat and another for TerraClimate—to understand how the data was retrieved and processed, and they can also generate their own output CSV files if needed. Using these pre-extracted CSV files, this notebook focuses on loading the predictor features and running the subsequent analysis and model training efficiently.
</p>
<p align="justify">
For more detailed guidance on the original data extraction process, you can review the <a href="https://planetarycomputer.microsoft.com/dataset/landsat-c2-l2#Example-Notebook">Landsat example notebook</a> and the <a href="https://planetarycomputer.microsoft.com/dataset/terraclimate#Example-Notebook">TerraClimate example notebook</a> available on the Planetary Computer portal.
</p>

<p align="justify">We have used selected spectral bands — SWIR22 (Shortwave Infrared 2), NIR (Near Infrared), Green, and SWIR16 (Shortwave Infrared 1) — and computed key spectral indices such as NDMI (Normalized Difference Moisture Index) and MNDWI (Modified Normalized Difference Water Index). These features capture surface moisture, vegetation, and water content characteristics that influence water quality variability. </p> <p align="justify"> In addition to Landsat features, we also incorporated the <b>Potential Evapotranspiration (PET)</b> variable from the <b>TerraClimate</b> dataset, which provides high-resolution global climate data. The PET feature captures the atmospheric demand for moisture, representing climatic conditions such as temperature, humidity, and radiation that influence surface water evaporation and thus affect water quality parameters. </p> <ul> <li>SWIR22 – Sensitive to surface moisture and turbidity variations in water bodies.</li> <li>NIR – Helps in identifying vegetation and suspended matter in water.</li> <li>Green – Useful for detecting water color and surface reflectance changes.</li> <li>SWIR16 – Provides information on surface dryness and sediment concentration.</li> <li>NDMI – Derived from NIR and SWIR16, indicates moisture and vegetation-water interaction.</li> <li>MNDWI – Derived from Green and SWIR22, effective for distinguishing open water areas and reducing built-up noise.</li> <li>PET – Extracted from the TerraClimate dataset, represents the potential evapotranspiration that influences hydrological and water quality dynamics.</li> </ul>

<h4 style="color:rgb(255, 0, 0)"><strong>Tip 1</strong></h4>  
<p align="justify">  
Participants are encouraged to experiment with different combinations of <b>Landsat</b> bands or even include data from other public satellite data sources. By creating mathematical combinations of bands, you can derive various spectral indices that capture surface and environmental characteristics. 
</p>


<h3>Loading Pre-Extracted Landsat Data</h3>
<p align="justify">
In this notebook, we <b>load previously extracted Landsat data</b> from CSV files generated in a separate extraction notebook. This approach ensures a smoother and faster workflow, allowing participants to focus on data analysis and model development without waiting for time-consuming data retrieval.
</p>
<p align="justify">
Participants are expected to generate their own data extraction CSV files by running the dedicated Landsat extraction notebook. These CSV files can then be used here to smoothly run this benchmark notebook. Participants can refer to the extraction notebook to understand the API-based process, including how individual bands and indices like <b>NDMI</b> were computed. Using these pre-extracted CSV files simplifies preprocessing and is ideal for large-scale environmental and water quality analysis.
</p>


<h4 style="color:rgb(255, 0, 0)"><strong>Tip 2</strong></h4>
In the data extraction process (performed in the dedicated extraction notebooks), a 100 m focal buffer was applied around each sampling location rather than using a single point. Participants may explore creating different focal buffers around the locations (e.g., 50 m, 150 m, etc.) during extraction. For example, if a 50 m buffer was used for “Band 2”, the extracted CSV values would reflect the average of "Band 2" within 50 meters of each location. This approach can help reduce errors associated with spatial autocorrelation.


In [18]:
landsat_train_features = pd.read_csv('../Data/landsat_features_training.csv')
landsat_train_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-28.760833,17.730278,02-01-2011,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595
1,-26.861111,28.884722,03-01-2011,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134
2,-26.450000,28.085833,03-01-2011,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805
3,-27.671111,27.236944,03-01-2011,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416
4,-27.356667,27.286389,03-01-2011,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683


<h3>Loading Pre-Extracted TerraClimate Data</h3>
<p align="justify">
In this notebook, we <b>load previously extracted TerraClimate data</b> from CSV files generated in a dedicated extraction notebook. This approach ensures a smoother and faster workflow, allowing participants to focus on data analysis and model development without waiting for time-consuming data retrieval.
</p>
<p align="justify">
Participants are expected to generate their own data extraction CSV files by running the dedicated TerraClimate extraction notebook. These CSV files can then be used here to smoothly run this benchmark notebook. Participants can refer to the extraction notebook to understand the API-based process, including how climate variables such as <b>Potential Evapotranspiration (PET)</b> were extracted. Using these pre-extracted CSV files ensures consistent, automated retrieval of high-resolution climate data that can be easily integrated with satellite-derived features for comprehensive environmental and hydrological analysis.
</p>


In [19]:
Terraclimate_df = pd.read_csv('../Data/terraclimate_features_training.csv')
Terraclimate_df.head()

,Latitude,Longitude,Sample Date,pet,ppt,tmax,soil,q,aet,def
0,-28.760833,17.730278,02-01-2011,236.3,8.3,35.76,0.0,0.4,7.9,228.40001
1,-26.861111,28.884722,03-01-2011,236.3,8.3,35.76,0.0,0.4,7.9,228.40001
2,-26.450000,28.085833,03-01-2011,174.2,32.7,36.10,0.0,1.6,31.1,143.10000
3,-27.671111,27.236944,03-01-2011,174.2,32.7,36.10,0.0,1.6,31.1,143.10000
4,-27.356667,27.286389,03-01-2011,190.7,21.4,36.21,0.0,1.1,20.4,170.30000


## Joining the predictor variables and response variables
Now that we have extracted our predictor variables, we need to join them onto the response variable . We use the function <i><b>combine_two_datasets</b></i> to combine the predictor variables and response variables.The <i><b>concat</b></i> function from pandas comes in handy here.

In [20]:
# Combine two datasets vertically (along columns) using pandas concat function.
def combine_two_datasets(dataset1,dataset2,dataset3):
    '''
    Returns a  vertically concatenated dataset.
    Attributes:
    dataset1 - Dataset 1 to be combined 
    dataset2 - Dataset 2 to be combined
    '''
    
    data = pd.concat([dataset1,dataset2,dataset3], axis=1)
    data = data.loc[:, ~data.columns.duplicated()]
    return data

In [21]:
# Combining ground data and final data into a single dataset.
wq_data = combine_two_datasets(Water_Quality_df, landsat_train_features, Terraclimate_df)
wq_data.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,nir,green,swir16,swir22,NDMI,MNDWI,pet,ppt,tmax,soil,q,aet,def
0,-28.760833,17.730278,02-01-2011,128.912,555.0,10.0,11190.0,11426.0,7687.5,7645.0,0.185538,0.195595,236.3,8.3,35.76,0.0,0.4,7.9,228.40001
1,-26.861111,28.884722,03-01-2011,74.720,162.9,163.0,17658.5,9550.0,13746.5,10574.0,0.124566,-0.180134,236.3,8.3,35.76,0.0,0.4,7.9,228.40001
2,-26.450000,28.085833,03-01-2011,89.254,573.0,80.0,15210.0,10720.0,17974.0,14201.0,-0.083293,-0.252805,174.2,32.7,36.10,0.0,1.6,31.1,143.10000
3,-27.671111,27.236944,03-01-2011,82.000,203.6,101.0,14887.0,10943.0,13522.0,11403.0,0.048048,-0.105416,174.2,32.7,36.10,0.0,1.6,31.1,143.10000
4,-27.356667,27.286389,03-01-2011,56.100,145.1,151.0,16828.5,9502.5,12665.5,9643.0,0.141147,-0.142683,190.7,21.4,36.21,0.0,1.1,20.4,170.30000


<h3>Handling Missing Values</h3>  
<p align="justify">  
Before model training, missing values in the dataset were carefully handled to ensure data consistency and prevent model bias. Numerical columns were imputed using their median values, maintaining the overall data distribution while minimizing the impact of outliers.  
</p>

In [22]:
# Landsat bands (nir, green, swir16, swir22, NDMI, MNDWI) have ~11.6% missing due to
# cloud cover. These will be imputed with spatial-temporal KNN after temporal features
# are added, so that month_sin/month_cos can be used as context for neighbour search.
# All other columns (TerraClimate etc.) have no missing values, but fill defensively.
LANDSAT_BANDS = ['nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI']

non_landsat = [c for c in wq_data.select_dtypes(include='number').columns
               if c not in LANDSAT_BANDS]
wq_data[non_landsat] = wq_data[non_landsat].fillna(wq_data[non_landsat].median())

print(f"Missing Landsat values (to be KNN-imputed): {wq_data[LANDSAT_BANDS].isna().sum().sum()}")
wq_data.isna().sum()

Missing Landsat values (to be KNN-imputed): 6510


Latitude                            0
Longitude                           0
Sample Date                         0
Total Alkalinity                    0
Electrical Conductance              0
Dissolved Reactive Phosphorus       0
nir                              1085
green                            1085
swir16                           1085
swir22                           1085
NDMI                             1085
MNDWI                            1085
pet                                 0
ppt                                 0
tmax                                0
soil                                0
q                                   0
aet                                 0
def                                 0
dtype: int64

### Creating Location Groups for Honest Evaluation

The training dataset has **162 unique locations but ~9,300 rows** — roughly 57 repeated measurements per site over 2011–2015. A random 70/30 shuffle puts the same physical riverbank in *both* train and test, so the model can memorise site-level patterns and report an inflated R². This is **data leakage**.

To get an honest estimate of how well the model generalises to *new* locations (which is exactly what the submission asks for), we use **location-grouped k-fold cross-validation** (`GroupKFold`). Every measurement from a given site is kept in the same fold — the held-out fold contains locations the model has never seen, mirroring the real submission task.

In [23]:
# Assign a unique integer group ID to every (Latitude, Longitude) pair.
# GroupKFold will use these to keep all measurements from the same site
# in the same fold — preventing location-level data leakage.
location_groups = pd.factorize(
    list(zip(wq_data['Latitude'], wq_data['Longitude']))
)[0]

print(f"Unique locations (groups): {len(set(location_groups))}")
print(f"Total rows              : {len(wq_data)}")
print(f"Avg measurements/site   : {len(wq_data) / len(set(location_groups)):.1f}")

Unique locations (groups): 162
Total rows              : 9319
Avg measurements/site   : 57.5


## Feature Engineering: Distance to Nearest City

> **Note — city-proximity features are defined below but excluded from the model.**  
> Although urbanisation influences water quality, distance/population metrics are geographic proxies that encode the training region's spatial layout. The validation set is in the Eastern Cape (East London / Port Elizabeth area), while training is dominated by the Johannesburg / Pretoria / Durban corridor — so these features carry no transferable signal and were found to cause overfitting to training geography.

In [24]:
# South African cities with coordinates and approximate populations.
# Sources: Statistics South Africa, municipal census estimates, Wikipedia.
sa_cities = pd.DataFrame({
    'City': [
        'Johannesburg', 'Cape Town', 'Durban', 'Pretoria', 'Port Elizabeth',
        'Pietermaritzburg', 'Bloemfontein', 'East London', 'Nelspruit',
        'Polokwane', 'Kimberley', 'Rustenburg', 'Vereeniging', 'Welkom',
        'Mahikeng', 'Klerksdorp', 'Witbank', 'Soweto', 'Tembisa',
        'Benoni', 'Boksburg', 'Midrand', 'Centurion', 'Mitchells Plain',
        'Paarl', 'George', 'Krugersdorp', 'Springs', 'Uitenhage',
        'Tzaneen', 'Thohoyandou', 'Musina', 'Upington', 'De Aar',
        'Beaufort West', 'Springbok', 'Stellenbosch', 'Mossel Bay',
        'Knysna', 'Oudtshoorn', 'Worcester', 'Richards Bay', 'Newcastle',
        'Ladysmith', 'Empangeni', 'Mthatha', 'Queenstown', 'Makhanda',
        'Phalaborwa', 'Lephalale',
    ],
    'Latitude': [
        -26.2041, -33.9249, -29.8587, -25.7479, -33.9608,
        -29.6006, -29.0852, -32.9958, -25.4745,
        -23.8962, -28.7381, -25.6672, -26.6732, -27.9931,
        -25.8553, -26.8544, -25.8771, -26.2665, -26.0023,
        -26.1845, -26.2143, -25.9942, -25.8601, -34.0478,
        -33.7342, -33.9628, -26.1010, -26.2626, -33.7578,
        -23.8305, -22.9467, -22.3390, -28.4478, -30.6458,
        -32.3507, -29.6638, -33.9321, -34.1800,
        -34.0348, -33.5900, -33.6461, -28.7831, -27.7573,
        -28.5691, -28.7272, -31.5893, -31.8998, -33.3042,
        -23.9406, -23.6728,
    ],
    'Longitude': [
        28.0473, 18.4241, 31.0218, 28.2293, 25.6022,
        30.3794, 26.1596, 27.9000, 30.9703,
        29.4486, 24.7701, 27.2427, 27.9261, 26.7354,
        25.6416, 26.6633, 29.2325, 27.8597, 28.2209,
        28.3186, 28.2598, 28.1236, 28.1674, 18.6299,
        18.9666, 22.4617, 27.7709, 28.4369, 25.3977,
        30.1625, 30.4853, 30.0494, 21.2561, 24.0121,
        22.5797, 17.8865, 18.8602, 22.1380,
        23.0472, 22.2060, 19.4481, 32.0390, 29.9341,
        29.7803, 31.9012, 28.7871, 26.8753, 26.5328,
        31.1426, 27.6966,
    ],
    'Population': [
        5635127, 4617560, 3720300, 3000000, 1307000,
         750845,  748000,  478676,  490000,
         500000,  250000,  549000,  120000,  400000,
         250000,  370000,  400000, 1695000,  700000,
         515000,  450000,  200000,  480000,  310000,
         170000,  200000,  340000,  250000,  200000,
         250000,  250000,   60000,  100000,   30000,
          40000,   20000,  160000,   90000,
          80000,   90000,  120000,  350000,  330000,
         230000,  120000,  250000,  200000,  100000,
         130000,  100000,
    ],
})

print(f"Loaded {len(sa_cities)} South African cities.")
sa_cities.head(10)

Loaded 50 South African cities.


,City,Latitude,Longitude,Population
0,Johannesburg,-26.2041,28.0473,5635127
1,Cape Town,-33.9249,18.4241,4617560
2,Durban,-29.8587,31.0218,3720300
3,Pretoria,-25.7479,28.2293,3000000
4,Port Elizabeth,-33.9608,25.6022,1307000
5,Pietermaritzburg,-29.6006,30.3794,750845
6,Bloemfontein,-29.0852,26.1596,748000
7,East London,-32.9958,27.9000,478676
8,Nelspruit,-25.4745,30.9703,490000
9,Polokwane,-23.8962,29.4486,500000


In [25]:
def add_city_features(df, cities_df):
    """
    For every sample in df, find the nearest South African city and compute:
      - dist_nearest_city_km  : Haversine distance to that city (km)
      - nearest_city_pop      : Population of that city
      - urban_influence       : Gravity score = population / distance^2
    """
    sample_lats = np.radians(df['Latitude'].values)
    sample_lons = np.radians(df['Longitude'].values)

    city_lats = np.radians(cities_df['Latitude'].values)
    city_lons = np.radians(cities_df['Longitude'].values)
    city_pops = cities_df['Population'].values

    R = 6371.0
    dlat = city_lats[None, :] - sample_lats[:, None]
    dlon = city_lons[None, :] - sample_lons[:, None]
    a = (np.sin(dlat / 2) ** 2
         + np.cos(sample_lats[:, None]) * np.cos(city_lats[None, :])
         * np.sin(dlon / 2) ** 2)
    distances_km = 2 * R * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

    nearest_idx  = np.argmin(distances_km, axis=1)
    nearest_dist = distances_km[np.arange(len(df)), nearest_idx]
    nearest_pop  = city_pops[nearest_idx]

    out = df.copy()
    out['dist_nearest_city_km'] = nearest_dist
    out['nearest_city_pop']     = nearest_pop.astype(float)
    out['urban_influence']      = nearest_pop / np.maximum(nearest_dist, 1.0) ** 2
    return out

# City-proximity features are NOT applied to the model training data —
# they caused geographic overfitting. The function is kept for reference.

## Additional Feature Engineering: Temporal / Seasonal Features

<p align="justify">
Water quality in South African rivers is strongly seasonal. The <b>wet season (October–March)</b>
brings heavy rainfall, increasing runoff and dilution, while the <b>dry season (April–September)</b>
concentrates pollutants. We encode this with four features:
</p>
<ul>
  <li><b>month_sin / month_cos</b> — Cyclical sine/cosine encoding of the calendar month so that
      December and January are treated as adjacent (no discontinuity at year-end).</li>
  <li><b>is_wet_season</b> — Binary flag: 1 for October–March, 0 otherwise.</li>
  <li><b>year</b> — Captures any multi-year trend in the 2011–2015 data.</li>
</ul>

In [26]:
# Parse the date column and extract temporal features.
wq_data['Sample Date'] = pd.to_datetime(wq_data['Sample Date'], dayfirst=True)
wq_data['month'] = wq_data['Sample Date'].dt.month
wq_data['year']  = wq_data['Sample Date'].dt.year

# Cyclical encoding — month 12 and month 1 are one step apart, not 11 apart.
wq_data['month_sin'] = np.sin(2 * np.pi * wq_data['month'] / 12)
wq_data['month_cos'] = np.cos(2 * np.pi * wq_data['month'] / 12)

# South Africa wet season: October–March (months 10, 11, 12, 1, 2, 3)
wq_data['is_wet_season'] = wq_data['month'].isin([10, 11, 12, 1, 2, 3]).astype(int)

print("Temporal features added.")
wq_data[['Sample Date', 'month', 'year', 'month_sin', 'month_cos', 'is_wet_season']].head(10)

Temporal features added.


,Sample Date,month,year,month_sin,month_cos,is_wet_season
0,2011-01-02,1,2011,0.5,0.866025,1
1,2011-01-03,1,2011,0.5,0.866025,1
2,2011-01-03,1,2011,0.5,0.866025,1
3,2011-01-03,1,2011,0.5,0.866025,1
4,2011-01-03,1,2011,0.5,0.866025,1
5,2011-01-04,1,2011,0.5,0.866025,1
6,2011-01-04,1,2011,0.5,0.866025,1
7,2011-01-04,1,2011,0.5,0.866025,1
8,2011-01-04,1,2011,0.5,0.866025,1
9,2011-01-04,1,2011,0.5,0.866025,1


## Model Building

<p align="justify"> Now let us select the columns required for our model building exercise. We will consider only Band swir22, NDMI and MNDWI from the Landsat data and pet from Terraclimate dataset as our predictor variables. It does not make sense to use latitude and longitude as predictor variables, as they do not have any direct impact on predicting the water quality parameters.</p>


In [27]:
# =============================================================================
# FEATURE ENGINEERING
# =============================================================================

# --- Spectral indices (original) ---
wq_data['NDVI'] = (wq_data['nir'] - wq_data['green']) / (wq_data['nir'] + wq_data['green'] + 1e-9)
wq_data['NIR_SWIR_ratio'] = wq_data['nir'] / (wq_data['swir22'] + 1e-9)

# --- Additional spectral indices ---
# NDWI (McFeeters): positive values indicate open water
wq_data['NDWI'] = (wq_data['green'] - wq_data['nir']) / (wq_data['green'] + wq_data['nir'] + 1e-9)
# Turbidity proxy: higher SWIR22 relative to NIR → more suspended sediment
wq_data['Turbidity'] = wq_data['swir22'] / (wq_data['nir'] + 1e-9)
# Band ratios capturing surface reflectance contrasts
wq_data['NIR_Green_ratio']    = wq_data['nir']    / (wq_data['green']  + 1e-9)
wq_data['SWIR_ratio']         = wq_data['swir16'] / (wq_data['swir22'] + 1e-9)
wq_data['Green_SWIR22_ratio'] = wq_data['green']  / (wq_data['swir22'] + 1e-9)
wq_data['NIR_SWIR22_ratio']   = wq_data['nir']    / (wq_data['swir22'] + 1e-9)

# --- Geographic proxy: distance to nearest South African coastline (km) ---
wq_data['Latitude']  = Water_Quality_df['Latitude']
wq_data['Longitude'] = Water_Quality_df['Longitude']

_coast_pts = np.array([
    [-34.36, 18.48],  # Cape Point
    [-33.93, 18.42],  # Cape Town
    [-34.18, 22.14],  # Mossel Bay
    [-33.98, 25.67],  # Port Elizabeth
    [-33.03, 27.91],  # East London
    [-31.59, 29.33],  # Coffee Bay
    [-29.86, 31.02],  # Durban
    [-27.77, 32.88],  # Richards Bay
    [-26.49, 32.95],  # Kosi Bay (Mozambique border area)
])

def dist_to_coast_km(lat, lon, coast=_coast_pts):
    dlat = coast[:, 0] - lat
    dlon = (coast[:, 1] - lon) * np.cos(np.radians(lat))
    return float(np.min(np.sqrt(dlat**2 + dlon**2)) * 111.0)

wq_data['dist_coast_km'] = wq_data.apply(
    lambda r: dist_to_coast_km(r['Latitude'], r['Longitude']), axis=1
)

# --- Climate-derived features ---
# Aridity index: PET / (rainfall + 1) — higher = drier = more concentrated minerals
wq_data['aridity']                = wq_data['pet'] / (wq_data['ppt'] + 1.0)
# Runoff coefficient: fraction of rainfall that becomes runoff → dilution signal
wq_data['runoff_coeff']           = wq_data['q']   / (wq_data['ppt'] + 1.0)
# Moisture deficit ratio: how much of potential evaporation is unfulfilled
wq_data['moisture_deficit_ratio'] = wq_data['def'] / (wq_data['pet'] + 1.0)
# AET/PET ratio: actual vs potential ET — water stress index (0–1)
wq_data['aet_pet_ratio']          = wq_data['aet'] / (wq_data['pet'] + 1.0)

# --- Interaction terms ---
# Latitude × tmax: temperature gradient with latitude is a key South Africa driver
wq_data['lat_tmax']      = wq_data['Latitude'] * wq_data['tmax']
# Aridity × NDVI: arid areas with sparse vegetation → different mineralogy
wq_data['aridity_ndvi']  = wq_data['aridity']  * wq_data['NDVI']
# Coast proximity × aridity: coastal arid zones have distinct ion chemistry
wq_data['coast_aridity'] = wq_data['dist_coast_km'] * wq_data['aridity']

# --- Final feature selection ---
FEATURE_COLS = [
    # Landsat bands
    'nir', 'green', 'swir16', 'swir22',
    # Original spectral indices
    'NDMI', 'MNDWI', 'NDVI', 'NIR_SWIR_ratio',
    # New spectral indices
    'NDWI', 'Turbidity', 'NIR_Green_ratio', 'SWIR_ratio', 'Green_SWIR22_ratio', 'NIR_SWIR22_ratio',
    # TerraClimate base
    'pet', 'ppt', 'tmax', 'soil', 'q', 'aet', 'def',
    # Climate-derived features
    'aridity', 'runoff_coeff', 'moisture_deficit_ratio', 'aet_pet_ratio',
    # Temporal
    'month_sin', 'month_cos', 'is_wet_season', 'year',
    # Geographic
    'Latitude', 'Longitude', 'dist_coast_km',
    # Interactions
    'lat_tmax', 'aridity_ndvi', 'coast_aridity',
]

TARGETS = ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']
wq_data = wq_data[FEATURE_COLS + TARGETS]

print(f"Feature set: {len(FEATURE_COLS)} predictors")
print(FEATURE_COLS)


Feature set: 35 predictors
['nir', 'green', 'swir16', 'swir22', 'NDMI', 'MNDWI', 'NDVI', 'NIR_SWIR_ratio', 'NDWI', 'Turbidity', 'NIR_Green_ratio', 'SWIR_ratio', 'Green_SWIR22_ratio', 'NIR_SWIR22_ratio', 'pet', 'ppt', 'tmax', 'soil', 'q', 'aet', 'def', 'aridity', 'runoff_coeff', 'moisture_deficit_ratio', 'aet_pet_ratio', 'month_sin', 'month_cos', 'is_wet_season', 'year', 'Latitude', 'Longitude', 'dist_coast_km', 'lat_tmax', 'aridity_ndvi', 'coast_aridity']


<h4 style="color:rgb(255, 0, 0)"><strong>Tip 3</strong></h4>
<p align="justify">We are developing individual models for each water quality parameter using a common set of features: SWIR22, NDMI, MNDWI, and PET. However, participants are encouraged to experiment with different feature combinations to build more robust machine learning models.</p>

## Helper Functions
### Location-Grouped Cross-Validation

<p align="justify">Instead of a random 70/30 train/test split, we use <b>GroupKFold</b> with groups defined by unique sampling locations. Each fold holds out a distinct set of locations that the model never sees during training — this directly mirrors the real submission task, where predictions are required for entirely new geographic sites.</p>

<p align="justify">After cross-validation reports honest generalisation metrics, a <b>final model is trained on all available data</b> (all 162 locations) to maximise the information available for the submission predictions.</p>

### Model Training
<p align="justify">We use <b>XGBoost</b> with moderate regularisation. Because the effective sample size is the number of unique locations (~162), not the number of rows (~9,300), the model is kept deliberately constrained to avoid memorising site-specific patterns.</p>

### Model Evaluation
<p align="justify">Performance is reported as the mean and standard deviation of R² and RMSE across the k location-held-out folds. A lower variance across folds indicates stable generalisation.</p>

<h4 style="color:rgb(255, 0, 0)"><strong>Tip 4</strong></h4>
<p align="justify">There are many data preprocessing methods available, which might help to improve the model performance. Participants should explore various suitable preprocessing methods as well as different machine learning algorithms to build a robust model.</p>

In [28]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

# ── Feature columns (geo-dependent variables excluded) ───────────────────────
# Latitude, Longitude, dist_coast_km, lat_tmax, coast_aridity are dropped:
# they encode training-region geography and carry no transferable signal to the
# Eastern Cape validation sites.
FEATURE_COLS = [
    # Landsat bands
    'nir', 'green', 'swir16', 'swir22',
    # Original spectral indices
    'NDMI', 'MNDWI', 'NDVI', 'NIR_SWIR_ratio',
    # New spectral indices
    'NDWI', 'Turbidity', 'NIR_Green_ratio', 'SWIR_ratio', 'Green_SWIR22_ratio', 'NIR_SWIR22_ratio',
    # TerraClimate base
    'pet', 'ppt', 'tmax', 'soil', 'q', 'aet', 'def',
    # Climate-derived features
    'aridity', 'runoff_coeff', 'moisture_deficit_ratio', 'aet_pet_ratio',
    # Temporal
    'month_sin', 'month_cos', 'is_wet_season', 'year',
    # Interactions (aridity_ndvi only; lat_tmax and coast_aridity are geo-dependent)
    'aridity_ndvi',
]

def train_model(X_train, y_train):
    """
    XGBoost with lighter regularisation.
    With only ~162 unique locations, max_depth=4 + lower lambda
    gives more signal without memorising site-level patterns.
    """
    model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def fit_target(X_tr, y_tr, name):
    """
    Fit XGBoost + Ridge on log1p-transformed target.
    Returns (xgb_model, ridge_model) — both trained on the same data.
    X_tr can be a numpy array or DataFrame.
    """
    y_log = np.log1p(y_tr)

    xgb_model = XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=2.0,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1,
    )
    xgb_model.fit(X_tr, y_log)

    ridge_model = Ridge(alpha=10.0)
    ridge_model.fit(X_tr, y_log)

    return xgb_model, ridge_model


def predict_target(X_va, xgb_model, ridge_model, name):
    """
    Ensemble XGBoost (70%) + Ridge (30%) predictions and back-transform from log1p.
    """
    xgb_pred   = xgb_model.predict(X_va)
    ridge_pred = ridge_model.predict(X_va)
    pred_log   = 0.7 * xgb_pred + 0.3 * ridge_pred
    return np.expm1(pred_log)


<div class="section">
  <h2>Model Workflow (Pipeline)</h2>
  <p align="justify">
    The complete model development process follows a structured pipeline to ensure consistency, reproducibility, and clarity. 
    Each stage in the workflow is modularized into independent functions, which can be reused for different water quality parameters. 
    This modular approach helps streamline the process and makes the workflow easily adaptable to new datasets or parameters in the future.
  </p>

  <p align="justify">
    The pipeline automates the sequence of steps — from data preparation to evaluation — for each target parameter. 
    The same set of predictor variables is used, while the response variable changes for each of the three targets: 
    <i>Total Alkalinity (TA)</i>, <i>Electrical Conductance (EC)</i>, and <i>Dissolved Reactive Phosphorus (DRP)</i>. 
    By maintaining a consistent framework, comparisons across models remain fair and interpretable.
  </p>


In [29]:
from sklearn.impute import SimpleImputer

def run_pipeline(X, y, groups, param_name="Parameter", log_transform=True, n_splits=5):
    print(f"\n{'='*60}")
    print(f"Training Model for {param_name}")
    if log_transform:
        print("  (target log1p-transformed; metrics on original scale)")
    print(f"{'='*60}")

    y_fit = np.log1p(y) if log_transform else y

    # Location-grouped cross-validation
    gkf = GroupKFold(n_splits=n_splits)
    cv_r2, cv_rmse = [], []
    ridge_r2 = []

    # Ridge pipeline: impute NaNs → scale → linear model
    ridge = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=10.0)),
    ])

    for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y_fit, groups=groups)):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y_fit.iloc[train_idx], y_fit.iloc[test_idx]

        fold_model = train_model(X_tr, y_tr)
        pred_fit   = fold_model.predict(X_te)

        pred_orig = np.expm1(pred_fit) if log_transform else pred_fit
        y_orig    = np.expm1(y_te)    if log_transform else y_te

        cv_r2.append(r2_score(y_orig, pred_orig))
        cv_rmse.append(np.sqrt(mean_squared_error(y_orig, pred_orig)))

        # Ridge baseline
        ridge.fit(X_tr, y_tr)
        rp = ridge.predict(X_te)
        rp_orig = np.expm1(rp) if log_transform else rp
        ridge_r2.append(r2_score(y_orig, rp_orig))

    print(f"\nGroupKFold CV ({n_splits} folds — each fold holds out distinct locations):")
    print(f"  XGBoost R²  : {np.mean(cv_r2):.3f}  ±  {np.std(cv_r2):.3f}")
    print(f"  XGBoost RMSE: {np.mean(cv_rmse):.3f}  ±  {np.std(cv_rmse):.3f}")
    print(f"  Ridge R²    : {np.mean(ridge_r2):.3f}  ±  {np.std(ridge_r2):.3f}  (linear baseline)")

    # Train final model on ALL data for submission
    final_model = train_model(X, y_fit)

    results = {
        "Parameter":   param_name,
        "R2_CV":       np.mean(cv_r2),
        "R2_CV_Std":   np.std(cv_r2),
        "RMSE_CV":     np.mean(cv_rmse),
        "RMSE_CV_Std": np.std(cv_rmse),
        "Ridge_R2_CV": np.mean(ridge_r2),
    }
    return final_model, pd.DataFrame([results])

In [30]:
def run_spatial_cv(X_scaled, y_TA, y_EC, y_DRP, groups, n_splits=5, drp_floor_mask=None):
    """
    Spatial cross-validation: GroupKFold holds out entire locations per fold.
    Fits independent XGBoost+Ridge ensembles for TA, EC, DRP on each fold.

    drp_floor_mask: boolean Series (True = row has a real DRP measurement,
                    False = DRP was at the detection limit of 10). When provided,
                    the DRP model is trained only on real measurements within
                    each fold's training split. TA and EC are unaffected.
    """
    kf = GroupKFold(n_splits=n_splits)
    fold_r2 = {'TA': [], 'EC': [], 'DRP': []}
    targets = {'TA': y_TA, 'EC': y_EC, 'DRP': y_DRP}

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X_scaled, y_TA, groups)):
        X_tr, X_va = X_scaled[tr_idx], X_scaled[va_idx]
        fold_scores = {}
        for name, y in targets.items():
            if name == 'DRP' and drp_floor_mask is not None:
                valid = drp_floor_mask.iloc[tr_idx].values
                X_tr_fit = X_tr[valid]
                y_tr_fit = y.iloc[tr_idx].values[valid]
            else:
                X_tr_fit = X_tr
                y_tr_fit = y.iloc[tr_idx].values

            xgb_model, ridge_model = fit_target(X_tr_fit, y_tr_fit, name)
            preds = predict_target(X_va, xgb_model, ridge_model, name)
            r2 = r2_score(y.iloc[va_idx], preds)
            fold_r2[name].append(r2)
            fold_scores[name] = r2

        avg = np.mean(list(fold_scores.values()))
        print(f"  Fold {fold+1}: TA={fold_scores['TA']:.3f}  EC={fold_scores['EC']:.3f}  DRP={fold_scores['DRP']:.3f}  avg={avg:.3f}")

    print(f"\n  Spatial CV means — TA: {np.mean(fold_r2['TA']):.3f} | EC: {np.mean(fold_r2['EC']):.3f} | DRP: {np.mean(fold_r2['DRP']):.3f}")
    overall = np.mean([np.mean(v) for v in fold_r2.values()])
    print(f"  Overall spatial CV R²: {overall:.3f}  ← leaderboard estimate")
    return fold_r2


print("Spatial CV defined.")

X = wq_data[FEATURE_COLS]
y_TA  = wq_data['Total Alkalinity']
y_EC  = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

groups = (
    Water_Quality_df['Latitude'].round(1).astype(str) + '_' +
    Water_Quality_df['Longitude'].round(1).astype(str)
)
print(f"Unique location groups: {groups.nunique()}")

# Rows where DRP was genuinely measured (not at the detection floor of 10)
drp_floor_mask = y_DRP != 10.0
print(f"DRP rows excluded from DRP training (floor=10): {(~drp_floor_mask).sum()} / {len(y_DRP)} ({100*(~drp_floor_mask).mean():.1f}%)")

# Impute NaN values before scaling — Ridge cannot handle NaN natively
from sklearn.impute import SimpleImputer
feat_imputer = SimpleImputer(strategy='median')
X_imputed = feat_imputer.fit_transform(X)

# Scale features — Ridge needs scaled input; XGBoost is unaffected but consistent
feat_scaler = StandardScaler()
X_scaled = feat_scaler.fit_transform(X_imputed)

# ── Spatial CV ──────────────────────────────────────────────────────────────
print("\nRunning 5-fold spatial CV (~10–20s)...")
cv_results = run_spatial_cv(X_scaled, y_TA, y_EC, y_DRP, groups,
                            drp_floor_mask=drp_floor_mask)

Spatial CV defined.
Unique location groups: 156
DRP rows excluded from DRP training (floor=10): 2558 / 9319 (27.4%)

Running 5-fold spatial CV (~10–20s)...
  Fold 1: TA=0.090  EC=-0.059  DRP=-0.019  avg=0.004
  Fold 2: TA=-0.057  EC=-0.092  DRP=-0.079  avg=-0.076
  Fold 3: TA=-0.329  EC=-0.277  DRP=-0.140  avg=-0.249
  Fold 4: TA=-0.205  EC=-0.118  DRP=-0.147  avg=-0.157
  Fold 5: TA=0.024  EC=-0.014  DRP=-0.003  avg=0.002

  Spatial CV means — TA: -0.095 | EC: -0.112 | DRP: -0.078
  Overall spatial CV R²: -0.095  ← leaderboard estimate


### Model Training and Evaluation for Each Parameter

<p align="justify">In this step, we apply the complete modeling pipeline to each of the three selected water quality parameters — Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus. The input feature set (<code>X</code>) remains the same across all three models, while the target variable (<code>y</code>) changes for each parameter. For every parameter, the <code>run_pipeline()</code> function is executed, which handles data preprocessing, model training, and both in-sample and out-of-sample evaluation. This ensures a consistent workflow and allows for a fair comparison of model performance across different water quality indicators.</p>

In [31]:
X = wq_data[FEATURE_COLS]

y_TA  = wq_data['Total Alkalinity']
y_EC  = wq_data['Electrical Conductance']
y_DRP = wq_data['Dissolved Reactive Phosphorus']

# All three targets log-transformed — TA is similarly right-skewed to EC and DRP
model_TA,  results_TA  = run_pipeline(X, y_TA,  location_groups, "Total Alkalinity",             log_transform=True)
model_EC,  results_EC  = run_pipeline(X, y_EC,  location_groups, "Electrical Conductance",        log_transform=True)
model_DRP, results_DRP = run_pipeline(X, y_DRP, location_groups, "Dissolved Reactive Phosphorus", log_transform=True)



Training Model for Total Alkalinity
  (target log1p-transformed; metrics on original scale)

GroupKFold CV (5 folds — each fold holds out distinct locations):
  XGBoost R²  : -0.149  ±  0.146
  XGBoost RMSE: 78.787  ±  9.747
  Ridge R²    : -0.134  ±  0.169  (linear baseline)

Training Model for Electrical Conductance
  (target log1p-transformed; metrics on original scale)

GroupKFold CV (5 folds — each fold holds out distinct locations):
  XGBoost R²  : -0.121  ±  0.117
  XGBoost RMSE: 357.516  ±  30.368
  Ridge R²    : -0.128  ±  0.107  (linear baseline)

Training Model for Dissolved Reactive Phosphorus
  (target log1p-transformed; metrics on original scale)

GroupKFold CV (5 folds — each fold holds out distinct locations):
  XGBoost R²  : -0.122  ±  0.063
  XGBoost RMSE: 53.278  ±  6.640
  Ridge R²    : -0.121  ±  0.075  (linear baseline)


### Model Performance Summary

<p align="justify">After training and evaluating the models for each water quality parameter, the individual performance metrics are combined into a single summary table. This table consolidates the R² and RMSE values for both in-sample and out-of-sample evaluations, enabling an easy comparison of model performance across Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus. Such a summary provides a quick overview of how well each model captures the variability in the respective parameter and highlights any differences in predictive accuracy.</p>

In [32]:
results_summary = pd.concat([results_TA, results_EC, results_DRP], ignore_index=True)
results_summary

,Parameter,R2_CV,R2_CV_Std,RMSE_CV,RMSE_CV_Std,Ridge_R2_CV
0,Total Alkalinity,-0.148777,0.145571,78.787347,9.747375,-0.134493
1,Electrical Conductance,-0.120561,0.117054,357.516306,30.367972,-0.127981
2,Dissolved Reactive Phosphorus,-0.122450,0.062582,53.278413,6.640133,-0.121437


## Submission

<p align="justify">Once you are satisfied with your model's performance, you can proceed to make predictions for unseen data. To do this, use your trained model to estimate the concentrations of the target water quality parameters — Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus — for the test locations provided in the "../Data/submission_template.csv" file. The same feature engineering pipeline (city proximity and temporal features) applied to the training data must be replicated here before generating predictions.</p>

In [33]:
# Reading the coordinates for the submission
test_file = pd.read_csv('../Data/submission_template.csv')
test_file.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN


In [34]:
landsat_val_features = pd.read_csv('../Data/landsat_features_validation.csv')
landsat_val_features.head()

,Latitude,Longitude,Sample Date,nir,green,swir16,swir22,NDMI,MNDWI
0,-32.043333,27.822778,01-09-2014,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052


In [35]:
Terraclimate_val_df = pd.read_csv('../Data/terraclimate_features_validation.csv')
Terraclimate_val_df.head()

,Latitude,Longitude,Sample Date,pet,ppt,tmax,soil,q,aet,def
0,-32.043333,27.822778,01-09-2014,131.2,101.0,25.369999,25.0,5.1,106.200005,25.0
1,-33.329167,26.077500,16-09-2015,123.8,29.0,27.250000,19.6,1.4,32.900000,90.9
2,-32.991639,27.640028,07-05-2015,122.4,51.8,27.109999,16.2,2.6,52.600002,69.8
3,-34.096389,24.439167,07-02-2012,84.0,68.3,23.880000,12.1,3.4,66.600000,17.4
4,-32.000556,28.581667,01-10-2014,84.5,146.1,20.580000,66.4,7.3,84.500000,0.0


In [36]:
# Consolidate all extracted features into a single validation dataframe
val_data = pd.DataFrame({
    'Longitude':   landsat_val_features['Longitude'].values,
    'Latitude':    landsat_val_features['Latitude'].values,
    'Sample Date': landsat_val_features['Sample Date'].values,
    # All Landsat bands
    'nir':    landsat_val_features['nir'].values,
    'green':  landsat_val_features['green'].values,
    'swir16': landsat_val_features['swir16'].values,
    'swir22': landsat_val_features['swir22'].values,
    # Landsat indices
    'NDMI':  landsat_val_features['NDMI'].values,
    'MNDWI': landsat_val_features['MNDWI'].values,
    # TerraClimate — all variables
    'pet':  Terraclimate_val_df['pet'].values,
    'ppt':  Terraclimate_val_df['ppt'].values,
    'tmax': Terraclimate_val_df['tmax'].values,
    'soil': Terraclimate_val_df['soil'].values,
    'q':    Terraclimate_val_df['q'].values,
    'aet':  Terraclimate_val_df['aet'].values,
    'def':  Terraclimate_val_df['def'].values,
})

In [37]:
# City-proximity features are excluded from the model — no add_city_features call needed.

In [38]:
# Apply the same temporal feature engineering as training
val_data['Sample Date'] = pd.to_datetime(val_data['Sample Date'], dayfirst=True)
val_data['month'] = val_data['Sample Date'].dt.month
val_data['year']  = val_data['Sample Date'].dt.year
val_data['month_sin']     = np.sin(2 * np.pi * val_data['month'] / 12)
val_data['month_cos']     = np.cos(2 * np.pi * val_data['month'] / 12)
val_data['is_wet_season'] = val_data['month'].isin([10, 11, 12, 1, 2, 3]).astype(int)

# Spectral indices (same as training)
val_data['NDVI']             = (val_data['nir'] - val_data['green']) / (val_data['nir'] + val_data['green'] + 1e-9)
val_data['NIR_SWIR_ratio']   = val_data['nir'] / (val_data['swir22'] + 1e-9)
val_data['NDWI']             = (val_data['green'] - val_data['nir']) / (val_data['green'] + val_data['nir'] + 1e-9)
val_data['Turbidity']        = val_data['swir22'] / (val_data['nir'] + 1e-9)
val_data['NIR_Green_ratio']  = val_data['nir']    / (val_data['green']  + 1e-9)
val_data['SWIR_ratio']       = val_data['swir16'] / (val_data['swir22'] + 1e-9)
val_data['Green_SWIR22_ratio'] = val_data['green'] / (val_data['swir22'] + 1e-9)
val_data['NIR_SWIR22_ratio'] = val_data['nir']    / (val_data['swir22'] + 1e-9)

# Distance to coastline (same coast points as training)
_coast_pts = np.array([
    [-34.36, 18.48], [-33.93, 18.42], [-34.18, 22.14],
    [-33.98, 25.67], [-33.03, 27.91], [-31.59, 29.33],
    [-29.86, 31.02], [-27.77, 32.88], [-26.49, 32.95],
])

def dist_to_coast_km(lat, lon, coast=_coast_pts):
    dlat = coast[:, 0] - lat
    dlon = (coast[:, 1] - lon) * np.cos(np.radians(lat))
    return float(np.min(np.sqrt(dlat**2 + dlon**2)) * 111.0)

val_data['dist_coast_km'] = val_data.apply(
    lambda r: dist_to_coast_km(r['Latitude'], r['Longitude']), axis=1
)

# Climate-derived features (same as training)
val_data['aridity']                = val_data['pet'] / (val_data['ppt'] + 1.0)
val_data['runoff_coeff']           = val_data['q']   / (val_data['ppt'] + 1.0)
val_data['moisture_deficit_ratio'] = val_data['def'] / (val_data['pet'] + 1.0)
val_data['aet_pet_ratio']          = val_data['aet'] / (val_data['pet'] + 1.0)

# Interaction terms (same as training)
val_data['lat_tmax']      = val_data['Latitude'] * val_data['tmax']
val_data['aridity_ndvi']  = val_data['aridity']  * val_data['NDVI']
val_data['coast_aridity'] = val_data['dist_coast_km'] * val_data['aridity']

print("All features engineered for validation data.")

All features engineered for validation data.


In [39]:
# Select the same feature columns used during training (geo-dependent features excluded)
# FEATURE_COLS is defined in the model-helpers cell — reused here for consistency.
submission_val_data = val_data[FEATURE_COLS].copy()
submission_val_data = submission_val_data.fillna(submission_val_data.median(numeric_only=True))
print(f"Validation features shape: {submission_val_data.shape}")
submission_val_data.head()


Validation features shape: (200, 30)


,nir,green,swir16,swir22,NDMI,MNDWI,NDVI,NIR_SWIR_ratio,NDWI,Turbidity,...,def,aridity,runoff_coeff,moisture_deficit_ratio,aet_pet_ratio,month_sin,month_cos,is_wet_season,year,aridity_ndvi
0,15229.0,12868.0,14797.0,12421.0,0.014388,-0.069727,0.084030,1.226069,-0.084030,0.815615,...,25.0,1.286275,0.050000,0.189107,0.803328,-1.000000,-1.836970e-16,0,2014,0.108086
1,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,0.205250,1.417217,-0.205250,0.705608,...,90.9,4.126667,0.046667,0.728365,0.263622,-1.000000,-1.836970e-16,0,2015,0.605938
2,16221.0,9304.5,12536.5,9958.0,0.128123,-0.147979,0.270964,1.628942,-0.270964,0.613896,...,69.8,2.318182,0.049242,0.565640,0.426256,0.500000,-8.660254e-01,0,2015,0.628145
3,14525.5,9493.5,12425.5,9973.0,0.081427,-0.130571,0.205250,1.417217,-0.205250,0.705608,...,17.4,1.212121,0.049062,0.204706,0.783529,0.866025,5.000000e-01,1,2012,0.605938
4,9125.0,11100.5,9455.0,8711.0,-0.017761,0.080052,-0.097674,1.047526,0.097674,0.954630,...,0.0,0.574439,0.049626,0.000000,0.988304,-0.866025,5.000000e-01,1,2014,-0.056108


In [40]:
# All three models trained on log1p scale → back-transform with expm1
pred_TA_submission  = np.expm1(model_TA.predict(submission_val_data))
pred_EC_submission  = np.expm1(model_EC.predict(submission_val_data))
pred_DRP_submission = np.expm1(model_DRP.predict(submission_val_data))

# Clip predictions to physically plausible ranges (no negatives)
pred_TA_submission  = np.clip(pred_TA_submission,  0, None)
pred_EC_submission  = np.clip(pred_EC_submission,  0, None)
pred_DRP_submission = np.clip(pred_DRP_submission, 0, None)

In [41]:
submission_df = pd.DataFrame({
    'Longitude':                     test_file['Longitude'].values,
    'Latitude':                      test_file['Latitude'].values,
    'Sample Date':                   test_file['Sample Date'].values,
    'Total Alkalinity':              pred_TA_submission,
    'Electrical Conductance':        pred_EC_submission,
    'Dissolved Reactive Phosphorus': pred_DRP_submission,
})
submission_df.head()

,Longitude,Latitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,27.822778,-32.043333,01-09-2014,122.748955,327.447601,26.392189
1,26.077500,-33.329167,16-09-2015,77.518677,370.147736,30.961426
2,27.640028,-32.991639,07-05-2015,78.198120,316.812408,22.567389
3,24.439167,-34.096389,07-02-2012,84.706581,316.100037,30.645147
4,28.581667,-32.000556,01-10-2014,105.227921,267.080322,13.055506


In [42]:
# Save predictions to submission.csv
submission_df.to_csv("submission.csv", index=False)
print("submission.csv saved successfully.")

submission.csv saved successfully.


### Upload submission file on platform

Upload the submission.csv on the <a href="https://challenge.ey.com">platform</a> to get a score generated on the scoreboard.

In [43]:
#Reading the coordinates for the submission
test_file = pd.read_csv('../Data/submission_template.csv')
test_file.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-32.043333,27.822778,01-09-2014,NaN,NaN,NaN
1,-33.329167,26.077500,16-09-2015,NaN,NaN,NaN
2,-32.991639,27.640028,07-05-2015,NaN,NaN,NaN
3,-34.096389,24.439167,07-02-2012,NaN,NaN,NaN
4,-32.000556,28.581667,01-10-2014,NaN,NaN,NaN
